In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset

In [3]:
df = pd.read_csv("/kaggle/input/datasets/anythingshashank/preprocess-dataset/preprocessed_data_lemma.csv")

In [4]:
df.head()

,text,label,word_count,lemmatized_text
0,valero energy corporation vlo is set report t...,negative,166,valero energy corporation vlo is set to report...
1,special feature form is sec does not requ...,neutral,145,a special feature of the form is that the sec ...
2,back may 2009 european union slammed chipmak...,negative,123,back in may 2009 the european union slammed ch...
3,instructions use magec rods — designed bra...,negative,156,instruction for use of the magec rod — designe...
4,mountain view calif ap google inc took step t...,positive,29,mountain view calif ap google inc took a step ...


In [5]:
df = df.drop(columns=["word_count"], axis=1)

In [6]:
df

,text,label,lemmatized_text
0,valero energy corporation vlo is set report t...,negative,valero energy corporation vlo is set to report...
1,special feature form is sec does not requ...,neutral,a special feature of the form is that the sec ...
2,back may 2009 european union slammed chipmak...,negative,back in may 2009 the european union slammed ch...
3,instructions use magec rods — designed bra...,negative,instruction for use of the magec rod — designe...
4,mountain view calif ap google inc took step t...,positive,mountain view calif ap google inc took a step ...
...,...,...,...
44995,summer sports direct acquired 51 stake pre...,neutral,in the summer sport direct acquired a 51 stake...
44996,revision came from amds slowerthananticipated...,negative,this revision came from amd slowerthananticipa...
44997,pointofsale technology provider payworks annou...,positive,pointofsale technology provider payworks annou...
44998,koei tecmo games is one japans more notable d...,neutral,koei tecmo game is one of japan more notable d...


In [7]:
le = LabelEncoder()
df["label_id"] = le.fit_transform(df["label"])

In [ ]:
df["word_count"] = df["text"].apply(lambda x: len(x.split()))
df["char_count"] = df["text"].apply(len)
df["avg_word_len"] = df["text"].apply(lambda x: np.mean([len(w) for w in x.split()]) if len(x.split()) > 0 else 0)

In [8]:
empty_count = (df["lemmatized_text"].str.strip() == "").sum()
print("Empty rows after cleaning:", empty_count)
print(df["lemmatized_text"].apply(lambda x: len(x.split())).describe())

Empty rows after cleaning: 0
count    45000.000000
mean        92.036222
std         42.186855
min          5.000000
25%         55.000000
50%         89.000000
75%        130.000000
max        762.000000
Name: lemmatized_text, dtype: float64


In [9]:
df.head()

,text,label,lemmatized_text,label_id
0,valero energy corporation vlo is set report t...,negative,valero energy corporation vlo is set to report...,0
1,special feature form is sec does not requ...,neutral,a special feature of the form is that the sec ...,1
2,back may 2009 european union slammed chipmak...,negative,back in may 2009 the european union slammed ch...,0
3,instructions use magec rods — designed bra...,negative,instruction for use of the magec rod — designe...,0
4,mountain view calif ap google inc took step t...,positive,mountain view calif ap google inc took a step ...,2


****Train/Test Split****

In [11]:
X = df["lemmatized_text"]
Y = df["label_id"]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2,random_state=42)

In [12]:
print(Y_train.value_counts())
print(X_train.index.equals(Y_train.index))

label_id
1    14140
2    13118
0     8742
Name: count, dtype: int64
True


In [13]:
X.head()

0    valero energy corporation vlo is set to report...
1    a special feature of the form is that the sec ...
2    back in may 2009 the european union slammed ch...
3    instruction for use of the magec rod — designe...
4    mountain view calif ap google inc took a step ...
Name: lemmatized_text, dtype: object

In [14]:
Y_train.value_counts()

label_id
1    14140
2    13118
0     8742
Name: count, dtype: int64

In [15]:
print(len(df), len(X_train), len(Y_train))
print(X_train.index.equals(Y_train.index))  # should be True

45000 36000 36000
True


In [16]:
print(X_train.index.equals(Y_train.index))

True


In [17]:
Y_test.value_counts()

label_id
1    3546
2    3240
0    2214
Name: count, dtype: int64

In [18]:
results = []

def record_result(model_name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    results.append({"model": model_name, "accuracy": round(acc, 4), "f1_macro": round(f1, 4)})
    print(f"{model_name} :- accuracy: {acc:.4f}, macro F1: {f1:.4f}")
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    print(cm)
    print(f"\n=== Classification Report ===")
    print(classification_report(y_true, y_pred, target_names=["negative", "neutral", "positive"]))

Classical ML Block

In [19]:
tfidf = TfidfVectorizer(max_features=2000, min_df=5, max_df=0.8, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)

TF-IDF train shape: (36000, 2000)
TF-IDF test shape: (9000, 2000)


In [20]:
print(tfidf.get_feature_names_out()[:20])
print(X_train_tfidf.shape[0] == len(Y_train))

['10' '100' '11' '12' '13' '14' '15' '16' '17' '18' '19' '20' '200' '2008'
 '2009' '2010' '2011' '2012' '2013' '2014']
True


In [21]:
nb_params = {"alpha": [0.5, 0.7, 1.0, 1.5, 2.0]}

nb_grid = GridSearchCV(
    MultinomialNB(),
    nb_params,
    cv=5,                     # 5-fold cross-validation instead of 3
    scoring="f1_macro",
    return_train_score=True   # lets us check for overfitting
)
nb_grid.fit(X_train_tfidf, Y_train)

print("Best alpha:", nb_grid.best_params_)
print("Best CV macro F1:", round(nb_grid.best_score_, 4))

# quick check: are we overfitting? compare train vs CV score for the best model
best_idx = nb_grid.best_index_
train_score = nb_grid.cv_results_["mean_train_score"][best_idx]
cv_score = nb_grid.cv_results_["mean_test_score"][best_idx]
print(f"Train F1: {train_score:.4f} | CV F1: {cv_score:.4f}")

nb_model = nb_grid.best_estimator_
nb_preds = nb_model.predict(X_test_tfidf)
record_result("Naive Bayes", Y_test, nb_preds)

Best alpha: {'alpha': 1.0}
Best CV macro F1: 0.6661
Train F1: 0.6792 | CV F1: 0.6661
Naive Bayes :- accuracy: 0.6682, macro F1: 0.6648
[[0.61878952 0.24119241 0.14001807]
 [0.11675127 0.67569092 0.20755781]
 [0.0904321  0.21574074 0.69382716]]

=== Classification Report ===
              precision    recall  f1-score   support

    negative       0.66      0.62      0.64      2214
     neutral       0.66      0.68      0.67      3546
    positive       0.68      0.69      0.69      3240

    accuracy                           0.67      9000
   macro avg       0.67      0.66      0.66      9000
weighted avg       0.67      0.67      0.67      9000



In [23]:
rf_params = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, 25],
    "min_samples_split": [4, 5, 6],
    "min_samples_leaf": [1, 2, 4] 
}

rf_grid = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_params,
    cv=5,                     # 5-fold cross-validation
    scoring="f1_macro",
    return_train_score=True,
    verbose=2
)
rf_grid.fit(X_train_tfidf, Y_train)

print("Best params:", rf_grid.best_params_)
print("Best CV macro F1:", round(rf_grid.best_score_, 4))

# overfitting check
best_idx = rf_grid.best_index_
train_score = rf_grid.cv_results_["mean_train_score"][best_idx]
cv_score = rf_grid.cv_results_["mean_test_score"][best_idx]
print(f"Train F1: {train_score:.4f} | CV F1: {cv_score:.4f}")

rf_model = rf_grid.best_estimator_
rf_preds = rf_model.predict(X_test_tfidf)
record_result("Random Forest", Y_test, rf_preds)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END max_depth=25, min_samples_leaf=2, min_samples_split=6, n_estimators=100; total time=   7.5s
[CV] END max_depth=25, min_samples_leaf=2, min_samples_split=6, n_estimators=100; total time=   7.5s
[CV] END max_depth=25, min_samples_leaf=2, min_samples_split=6, n_estimators=100; total time=   7.3s
[CV] END max_depth=25, min_samples_leaf=2, min_samples_split=6, n_estimators=100; total time=   7.3s
[CV] END max_depth=25, min_samples_leaf=2, min_samples_split=6, n_estimators=100; total time=   7.4s
[CV] END max_depth=25, min_samples_leaf=2, min_samples_split=4, n_estimators=200; total time=  14.7s
[CV] END max_depth=25, min_samples_leaf=2, min_samples_split=4, n_estimators=200; total time=  15.3s
[CV] END max_depth=25, min_samples_leaf=2, min_samples_split=4, n_estimators=200; total time=  14.8s
[CV] END max_depth=25, min_samples_leaf=2, min_samples_split=4, n_estimators=200; total time=  15.0s
[CV] END max_depth=25, min_sam

In [ ]:
xgb_params = {
    "n_estimators": [200, 300],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1, 0.2],
    "subsample": [0.8, 1.0]
}

xgb_grid = RandomizedSearchCV(
    XGBClassifier(
        random_state=42,
        n_jobs=-1,
        tree_method="hist",
        device="cuda"
        # T4 GPU
    ),
    xgb_params,
    n_iter=10,
    cv=5,                     # 5-fold cross-validation
    scoring="f1_macro",
    return_train_score=True,
    n_jobs=1,
    verbose=2                 # prints progress per fit so you can see it's alive
)
xgb_grid.fit(X_train_tfidf, Y_train)

print("Best params:", xgb_grid.best_params_)
print("Best CV macro F1:", round(xgb_grid.best_score_, 4))

# overfitting check
best_idx = xgb_grid.best_index_
train_score = xgb_grid.cv_results_["mean_train_score"][best_idx]
cv_score = xgb_grid.cv_results_["mean_test_score"][best_idx]
print(f"Train F1: {train_score:.4f} | CV F1: {cv_score:.4f}")

xgb_model = xgb_grid.best_estimator_
xgb_preds = xgb_model.predict(X_test_tfidf)
record_result("XGBoost", Y_test, xgb_preds)

In [24]:
VOCAB_SIZE = 5000
MAX_LEN = 40  # covers the vast majority of sentence lengths seen in the EDA histogram

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post")

print("Padded train shape:", X_train_pad.shape)


# GPU sanity check (TF sometimes falls back to CPU silently)
print("GPU devices:", tf.config.list_physical_devices('GPU'))

Padded train shape: (36000, 40)
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [25]:
def build_bilstm(embedding_dim, lstm_units, learning_rate):
    model = Sequential([
        Embedding(input_dim=VOCAB_SIZE, output_dim=embedding_dim, input_length=MAX_LEN),
        Bidirectional(LSTM(lstm_units)),
        Dropout(0.3),
        Dense(32, activation="relu"),
        Dense(3, activation="softmax")
    ])
    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

In [26]:
# try a couple of simple configs and keep the best one — no full sweep, just a small manual search
configs = [
    {"embedding_dim": 64, "lstm_units": 32, "learning_rate": 0.001},
    {"embedding_dim": 100, "lstm_units": 64, "learning_rate": 0.001},
    {"embedding_dim": 100, "lstm_units": 64, "learning_rate": 0.0005},
]

best_f1 = 0
best_model = None
best_config = None

for cfg in configs:
    print("Trying config:", cfg)
    model = build_bilstm(**cfg)
    model.fit(
        X_train_pad, Y_train,
        validation_split=0.1,
        epochs=8,
        batch_size=32,
        verbose=1
    )
    preds = np.argmax(model.predict(X_test_pad, verbose=0), axis=1)
    f1 = f1_score(Y_test, preds, average="macro")
    print("macro F1:", round(f1, 4))

    if f1 > best_f1:
        best_f1 = f1
        best_model = model
        best_config = cfg

print("\nBest config:", best_config)

Trying config: {'embedding_dim': 64, 'lstm_units': 32, 'learning_rate': 0.001}


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1786475565.011967   10733 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13734 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5


Epoch 1/8


I0000 00:00:1786475565.014176   10733 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


1013/1013 ━━━━━━━━━━━━━━━━━━━━ 14s 9ms/step - accuracy: 0.5894 - loss: 0.8622 - val_accuracy: 0.6483 - val_loss: 0.7948
Epoch 2/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.6934 - loss: 0.7110 - val_accuracy: 0.6436 - val_loss: 0.7840
Epoch 3/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.7258 - loss: 0.6419 - val_accuracy: 0.6333 - val_loss: 0.8273
Epoch 4/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.7654 - loss: 0.5621 - val_accuracy: 0.6275 - val_loss: 0.8984
Epoch 5/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.8070 - loss: 0.4780 - val_accuracy: 0.6156 - val_loss: 0.9717
Epoch 6/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.8424 - loss: 0.3957 - val_accuracy: 0.6000 - val_loss: 1.1306
Epoch 7/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.8742 - loss: 0.3223 - val_accuracy: 0.5911 - val_loss: 1.3049
Epoch 8/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.8986 - loss: 0.2625 - val_accuracy: 0.5

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1013/1013 ━━━━━━━━━━━━━━━━━━━━ 11s 9ms/step - accuracy: 0.5995 - loss: 0.8579 - val_accuracy: 0.6444 - val_loss: 0.7863
Epoch 2/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.6942 - loss: 0.7097 - val_accuracy: 0.6447 - val_loss: 0.8031
Epoch 3/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.7323 - loss: 0.6281 - val_accuracy: 0.6400 - val_loss: 0.8394
Epoch 4/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.7882 - loss: 0.5182 - val_accuracy: 0.6094 - val_loss: 0.9325
Epoch 5/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.8360 - loss: 0.4143 - val_accuracy: 0.6008 - val_loss: 1.0688
Epoch 6/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.8770 - loss: 0.3164 - val_accuracy: 0.5894 - val_loss: 1.3078
Epoch 7/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.9090 - loss: 0.2388 - val_accuracy: 0.5803 - val_loss: 1.5105
Epoch 8/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.9330 - loss: 0.1792 - val_accuracy: 0.5

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1013/1013 ━━━━━━━━━━━━━━━━━━━━ 11s 9ms/step - accuracy: 0.5891 - loss: 0.8625 - val_accuracy: 0.6386 - val_loss: 0.7960
Epoch 2/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.6955 - loss: 0.7084 - val_accuracy: 0.6475 - val_loss: 0.7955
Epoch 3/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.7194 - loss: 0.6617 - val_accuracy: 0.6364 - val_loss: 0.8192
Epoch 4/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 9ms/step - accuracy: 0.7446 - loss: 0.6066 - val_accuracy: 0.6322 - val_loss: 0.8301
Epoch 5/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.7758 - loss: 0.5338 - val_accuracy: 0.6258 - val_loss: 0.9090
Epoch 6/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.8148 - loss: 0.4505 - val_accuracy: 0.6022 - val_loss: 1.0379
Epoch 7/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.8505 - loss: 0.3768 - val_accuracy: 0.6025 - val_loss: 1.2298
Epoch 8/8
1013/1013 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - accuracy: 0.8784 - loss: 0.3043 - val_accuracy: 0.5

In [27]:
bilstm_preds = np.argmax(best_model.predict(X_test_pad, verbose=0), axis=1)
record_result("BiLSTM", Y_test, bilstm_preds)

BiLSTM :- accuracy: 0.5993, macro F1: 0.6002
[[0.63504968 0.21183379 0.15311653]
 [0.17428088 0.58516638 0.24055274]
 [0.125      0.2845679  0.5904321 ]]

=== Classification Report ===
              precision    recall  f1-score   support

    negative       0.58      0.64      0.61      2214
     neutral       0.60      0.59      0.59      3546
    positive       0.62      0.59      0.60      3240

    accuracy                           0.60      9000
   macro avg       0.60      0.60      0.60      9000
weighted avg       0.60      0.60      0.60      9000



In [21]:
df_train = pd.DataFrame({"text": X_train.index.map(lambda i: df.loc[i, "text"]), "label": Y_train.values})
df_test = pd.DataFrame({"text": X_test.index.map(lambda i: df.loc[i, "text"]), "label": Y_test.values})

train_hf = Dataset.from_pandas(df_train.reset_index(drop=True))
test_hf = Dataset.from_pandas(df_test.reset_index(drop=True))

MODEL_NAME = "distilbert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return bert_tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)

train_hf = train_hf.map(tokenize_batch, batched=True)
test_hf = test_hf.map(tokenize_batch, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/36000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

In [22]:
bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

training_args = TrainingArguments(
    output_dir="./bert_results",
    num_train_epochs=3,          # light tuning: 3 epochs works well for small datasets like this
    learning_rate=2e-5,           # standard starting point for fine-tuning DistilBERT
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    logging_strategy="epoch",
    report_to="none",
    fp16=True,             # ~2x speedup on T4 GPU
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=test_hf,
    compute_metrics=compute_metrics
)

trainer.train()

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, 

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.237764,1.091633,0.773222,0.773204
2,0.902603,1.067695,0.784667,0.785911
3,0.690361,1.163365,0.782889,0.784320


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=3375, training_loss=0.9435759910300926, metrics={'train_runtime': 572.428, 'train_samples_per_second': 188.67, 'train_steps_per_second': 5.896, 'total_flos': 1788341773824000.0, 'train_loss': 0.9435759910300926, 'epoch': 3.0})

In [30]:
bert_eval = trainer.evaluate()
results.append({
    "model": "DistilBERT",
    "accuracy": round(bert_eval["eval_accuracy"], 4),
    "f1_macro": round(bert_eval["eval_f1_macro"], 4)
})
print(bert_eval)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 1.1626187562942505, 'eval_accuracy': 0.7854444444444444, 'eval_f1_macro': 0.7862224913682428, 'eval_runtime': 14.3831, 'eval_samples_per_second': 625.733, 'eval_steps_per_second': 19.606, 'epoch': 3.0}


In [ ]:
# Do Change 2 first (FinBERT swap) alone.
# Keep epochs=3, learning_rate=2e-5 exactly as before.
# Run it.
# Note the new eval_f1_macro.
# This isolates: did switching to a domain-specific model help?

In [24]:
df_train = pd.DataFrame({"lemmatized_text": X_train.index.map(lambda i: df.loc[i, "lemmatized_text"]), "label": Y_train.values})
df_test = pd.DataFrame({"lemmatized_text": X_test.index.map(lambda i: df.loc[i, "lemmatized_text"]), "label": Y_test.values})

train_hf = Dataset.from_pandas(df_train.reset_index(drop=True))
test_hf = Dataset.from_pandas(df_test.reset_index(drop=True))

MODEL_NAME = "yiyanghkust/finbert-tone"
bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return bert_tokenizer(batch["lemmatized_text"], truncation=True, padding="max_length", max_length=64)

train_hf = train_hf.map(tokenize_batch, batched=True)
test_hf = test_hf.map(tokenize_batch, batched=True)


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/36000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

In [25]:
bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

training_args = TrainingArguments(
    output_dir="./bert_results",
    num_train_epochs=3,          # light tuning: 3 epochs works well for small datasets like this
    learning_rate=2e-5,           # standard starting point for fine-tuning DistilBERT
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    logging_strategy="epoch",
    report_to="none",
    fp16=True,             # ~2x speedup on T4 GPU
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=test_hf,
    compute_metrics=compute_metrics
)

trainer.train()

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: yiyanghkust/finbert-tone
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=3375, training_loss=0.9825522099247685, metrics={'train_runtime': 1120.3538, 'train_samples_per_second': 96.398, 'train_steps_per_second': 3.012, 'total_flos': 3552031139328000.0, 'train_loss': 0.9825522099247685, 'epoch': 3.0})

In [26]:
bert_eval = trainer.evaluate()
results.append({
    "model": "FinBERT",
    "accuracy": round(bert_eval["eval_accuracy"], 4),
    "f1_macro": round(bert_eval["eval_f1_macro"], 4)
})
print(bert_eval)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 1.2879133224487305, 'eval_accuracy': 0.7724444444444445, 'eval_f1_macro': 0.7727483939950276, 'eval_runtime': 27.0957, 'eval_samples_per_second': 332.156, 'eval_steps_per_second': 10.408, 'epoch': 3.0}


In [27]:
bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

training_args = TrainingArguments(
    output_dir="./bert_results",
    num_train_epochs=10,          # light tuning: 3 epochs works well for small datasets like this
    learning_rate=3e-5,           # standard starting point for fine-tuning DistilBERT
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    logging_strategy="epoch",
    report_to="none",
    fp16=True,             # ~2x speedup on T4 GPU
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=test_hf,
    compute_metrics=compute_metrics
)
trainer.train()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: yiyanghkust/finbert-tone
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.412174,1.299167,0.743222,0.736546
2,0.890986,1.161266,0.771000,0.771270
3,0.464625,1.617534,0.761444,0.761027
4,0.206430,2.498138,0.763000,0.763732
5,0.124792,4.123758,0.745778,0.747792
6,0.081530,4.377081,0.767444,0.768103


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: [enforce fail at inline_container.cc:668] . unexpected pos 740277376 vs 740277268